# 04b · Preprocesamiento — versión MENSUAL (pipeline paralelo)
**Tesis:** Medición del ciclo financiero en Perú mediante técnicas de reducción dimensional y machine learning

**Por qué existe este notebook aparte de `04_preprocesamiento.ipynb`:**
Con 72 trimestres (18 años), la restricción n > p limita mucho el tamaño de
ventana viable para EPCA/EHFC (ver `06a_seleccion_ventana_epca.ipynb`).
En mensual se dispone de ~216 observaciones en vez de 72, lo que permite
ventanas más cortas en tiempo calendario sin caer en matrices de
correlación degeneradas.

**Diferencia clave frente a la versión trimestral: NO incluye
`indice_precios_inmuebles`.** Esa serie es trimestral en origen (no existe
una versión mensual real publicada por el BCRP); incluirla aquí obligaría
a interpolar 2 de cada 3 meses, es decir, fabricar información que no
existe. Se prefiere excluirla explícitamente antes que inventar datos.
Esto se documenta como decisión metodológica reconocida, no oculta.

**Entradas (solo lectura):**
- `data/raw/series_bcrp_raw.csv` (ya mensual, sin resamplear)
- `data/raw/embi_peru_raw.csv` (diario, se resamplea aquí a MENSUAL, no trimestral)

**Salidas:** cuatro datasets en `data/processed/`, sufijo `_mensual`, en
paralelo exacto a los cuatro de la versión trimestral.

## 1. Librerías y carga de datos crudos (solo lectura)

In [1]:
import os
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df_raw = pd.read_csv('data/raw/series_bcrp_raw.csv', index_col=0, parse_dates=True)
df_raw = df_raw.sort_index()
print(f'Series crudas mensuales: {df_raw.shape[0]} meses x {df_raw.shape[1]} variables')
print(f'Periodo: {df_raw.index.min().strftime("%Y-%m")} a {df_raw.index.max().strftime("%Y-%m")}')

Series crudas mensuales: 228 meses x 18 variables
Periodo: 2007-01 a 2025-12


## 2. EMBI Perú: resampleo a MENSUAL (no trimestral)

Mismo criterio de promedio del periodo que en la versión trimestral, pero
agregando a nivel mes.

In [2]:
df_embi_diario = pd.read_csv('data/raw/embi_peru_raw.csv', index_col=0, parse_dates=True)
df_embi_mensual = df_embi_diario.resample('MS').mean()
print(f'EMBI Perú (resampleado a mensual): {df_embi_mensual.shape[0]} observaciones')

EMBI Perú (resampleado a mensual): 276 observaciones


## 3. Unir EMBI con el resto de series mensuales

In [3]:
df_completo = df_raw.join(df_embi_mensual, how='inner')
print(f'Dataset mensual sincronizado: {df_completo.shape[0]} filas x {df_completo.shape[1]} columnas')
print(f'Periodo: {df_completo.index.min().strftime("%Y-%m")} a {df_completo.index.max().strftime("%Y-%m")}')
df_completo.tail()

Dataset mensual sincronizado: 228 filas x 19 columnas
Periodo: 2007-01 a 2025-12


,Crédito SF sector privado,Crédito empresarial,Crédito consumo,Crédito hipotecario,Tasa activa TAMN,Tasa pasiva TIPMN,Tasa referencia BCRP,Tipo de cambio,Índice BVL,IPC Lima,PBI desestacionalizado,Demanda interna,Reservas internacionales,Liquidez M1,Liquidez M2,Liquidez M3,Tasa interbancaria,IPC Subyacente,embi_peru
fecha,,,,,,,,,,,,,,,,,,,
2025-08-01,470356.301562,247323.020900,104750.3996,72784.2576,15.1490,2.1348,4.50,3.543200,34937.73,115.586828,191.100818,216.143896,87752.818921,158984.423067,516153.852177,702940.601847,4.5079,115.998679,133.095238
2025-09-01,470208.838774,246752.668765,105563.9020,73326.4468,15.3507,2.1383,4.25,3.502818,38054.76,115.599718,191.867835,210.518623,85148.225305,161111.064693,520987.904396,709175.882797,4.3685,116.016556,NaN
2025-10-01,470552.782777,247446.294924,106685.7408,73902.4976,15.5555,2.1200,4.25,3.414091,38599.28,115.484549,192.028747,217.057443,89960.765240,163534.940316,525748.900702,712922.060965,4.2500,116.070864,123.173913
2025-11-01,474468.883319,249857.815330,107487.5164,74421.5588,15.7477,2.0993,4.25,3.372850,38978.53,115.612485,190.919048,215.534038,90898.183584,170252.343551,534918.802279,721826.606486,4.2478,116.220266,126.100000
2025-12-01,479089.326795,252432.680596,107850.7452,74632.2836,15.8871,1.9771,4.25,3.366737,43464.77,115.894705,193.429032,242.172683,90214.484300,178926.594941,550978.611986,737697.228516,4.2411,116.492958,126.782609


## 4. Revisión y tratamiento de nulos

Mismo criterio que la versión trimestral: se documenta cuántas filas se
pierden antes de eliminarlas, no se hace en silencio.

In [4]:
nulos_por_columna = df_completo.isna().sum()
print('Valores faltantes por columna:')
print(nulos_por_columna[nulos_por_columna > 0])

filas_antes = df_completo.shape[0]
df_limpio = df_completo.dropna()
filas_despues = df_limpio.shape[0]

print(f'\nFilas antes de dropna: {filas_antes}')
print(f'Filas despues de dropna: {filas_despues}')
print(f'Filas eliminadas: {filas_antes - filas_despues}')
print(f'Periodo final: {df_limpio.index.min().strftime("%Y-%m")} a {df_limpio.index.max().strftime("%Y-%m")}')

Valores faltantes por columna:
embi_peru    19
dtype: int64

Filas antes de dropna: 228
Filas despues de dropna: 209
Filas eliminadas: 19
Periodo final: 2007-01 a 2025-12


## 5. Versión niveles + estandarización (SIN transformar)

Igual que la sección 4b de la versión trimestral: permite comparar PCA/HFC
sobre niveles vs. transformado, ahora en frecuencia mensual.

In [5]:
scaler_niveles = StandardScaler()
df_niveles_estandarizado = pd.DataFrame(
    scaler_niveles.fit_transform(df_limpio),
    index=df_limpio.index,
    columns=df_limpio.columns
)
print('Verificación — media ≈ 0, varianza ≈ 1 (niveles estandarizados, mensual):')
df_niveles_estandarizado.describe().loc[['mean', 'std']].round(3)

Verificación — media ≈ 0, varianza ≈ 1 (niveles estandarizados, mensual):


,Crédito SF sector privado,Crédito empresarial,Crédito consumo,Crédito hipotecario,Tasa activa TAMN,Tasa pasiva TIPMN,Tasa referencia BCRP,Tipo de cambio,Índice BVL,IPC Lima,PBI desestacionalizado,Demanda interna,Reservas internacionales,Liquidez M1,Liquidez M2,Liquidez M3,Tasa interbancaria,IPC Subyacente,embi_peru
mean,0.000,-0.000,-0.000,0.000,0.000,0.000,-0.000,-0.000,0.000,-0.000,0.000,-0.000,0.000,0.000,-0.000,0.000,0.000,-0.000,0.000
std,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002


## 6. Transformaciones de estacionariedad (log-diferencia / diferencia simple)

Mismo criterio ya sustentado (Drehmann, Borio & Tsatsaronis, 2012;
Nelson & Plosser, 1982): log-diferencia para niveles, diferencia simple
para tasas y spreads. Idéntica clasificación de variables que en la
versión trimestral, sin `indice_precios_inmuebles` (no existe aquí).

In [6]:
VARS_LOG_DIFF = [
    # 'Crédito SF sector privado',  # Total -- excluir por colinealidad, igual que en trimestral
    'Crédito empresarial', 'Crédito consumo', 'Crédito hipotecario',
    'Tipo de cambio', 'Índice BVL', 'IPC Lima',
    'PBI desestacionalizado', 'Demanda interna', 'Reservas internacionales',
    'Liquidez M1', 'Liquidez M2', 'Liquidez M3',
]
VARS_DIFF_SIMPLE = [
    'Tasa activa TAMN', 'Tasa pasiva TIPMN', 'Tasa referencia BCRP',
    'Tasa interbancaria',
    'embi_peru',
]
# 'IPC Subyacente' queda fuera, misma decision pendiente que en la version trimestral.

df_transformado = pd.DataFrame(index=df_limpio.index)

for col in VARS_LOG_DIFF:
    if col in df_limpio.columns:
        df_transformado[f'{col}_log_diff'] = np.log(df_limpio[col]).diff()

for col in VARS_DIFF_SIMPLE:
    if col in df_limpio.columns:
        df_transformado[f'{col}_diff'] = df_limpio[col].diff()

df_transformado = df_transformado.dropna()
print(f'Dataset transformado (mensual, estacionario): {df_transformado.shape[0]} filas x {df_transformado.shape[1]} columnas')
df_transformado.tail()

Dataset transformado (mensual, estacionario): 208 filas x 17 columnas


,Crédito empresarial_log_diff,Crédito consumo_log_diff,Crédito hipotecario_log_diff,Tipo de cambio_log_diff,Índice BVL_log_diff,IPC Lima_log_diff,PBI desestacionalizado_log_diff,Demanda interna_log_diff,Reservas internacionales_log_diff,Liquidez M1_log_diff,Liquidez M2_log_diff,Liquidez M3_log_diff,Tasa activa TAMN_diff,Tasa pasiva TIPMN_diff,Tasa referencia BCRP_diff,Tasa interbancaria_diff,embi_peru_diff
fecha,,,,,,,,,,,,,,,,,
2025-07-01,0.008822,0.004928,0.002894,-0.013325,0.021602,0.002312,0.011317,0.051459,0.025666,0.029998,0.022872,0.023006,0.0638,-0.1121,0.00,0.001106,-13.134576
2025-08-01,0.006494,0.008378,0.007042,-0.003662,0.043806,-0.002907,0.004533,-0.003674,0.003114,-0.002453,0.005962,0.003252,0.1045,-0.0268,0.00,0.007900,-7.817805
2025-10-01,0.000498,0.018307,0.015247,-0.037119,0.099666,-0.000885,0.004844,0.004218,0.024850,0.028220,0.018419,0.014100,0.4065,-0.0148,-0.25,-0.257900,-9.921325
2025-11-01,0.009698,0.007487,0.006999,-0.012153,0.009777,0.001107,-0.005796,-0.007043,0.010366,0.040255,0.017291,0.012413,0.1922,-0.0207,0.00,-0.002200,2.926087
2025-12-01,0.010253,0.003374,0.002828,-0.001814,0.108940,0.002438,0.013061,0.116532,-0.007550,0.049694,0.029581,0.021749,0.1394,-0.1222,0.00,-0.006700,0.682609


## 7. Estandarización de la versión transformada

In [7]:
scaler_transf = StandardScaler()
df_estandarizado = pd.DataFrame(
    scaler_transf.fit_transform(df_transformado),
    index=df_transformado.index,
    columns=df_transformado.columns
)
print('Verificación — media ≈ 0, varianza ≈ 1 (transformado + estandarizado, mensual):')
df_estandarizado.describe().loc[['mean', 'std']].round(3)

Verificación — media ≈ 0, varianza ≈ 1 (transformado + estandarizado, mensual):


,Crédito empresarial_log_diff,Crédito consumo_log_diff,Crédito hipotecario_log_diff,Tipo de cambio_log_diff,Índice BVL_log_diff,IPC Lima_log_diff,PBI desestacionalizado_log_diff,Demanda interna_log_diff,Reservas internacionales_log_diff,Liquidez M1_log_diff,Liquidez M2_log_diff,Liquidez M3_log_diff,Tasa activa TAMN_diff,Tasa pasiva TIPMN_diff,Tasa referencia BCRP_diff,Tasa interbancaria_diff,embi_peru_diff
mean,-0.000,0.000,0.000,0.000,0.000,-0.000,0.000,0.000,-0.000,-0.000,0.000,0.000,0.000,-0.000,0.000,0.000,0.000
std,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002,1.002


## 8. Guardar los cuatro datasets mensuales

Mismo esquema de nombres que la versión trimestral, con sufijo `_mensual`
para que ambos pipelines convivan sin sobrescribirse.

In [8]:
os.makedirs('data/processed', exist_ok=True)

df_limpio.to_csv('data/processed/dataset_niveles_mensual.csv')
df_niveles_estandarizado.to_csv('data/processed/dataset_niveles_estandarizado_mensual.csv')
df_transformado.to_csv('data/processed/dataset_transformado_mensual.csv')
df_estandarizado.to_csv('data/processed/dataset_estandarizado_mensual.csv')

print('Datasets guardados en data/processed/:')
print('  - dataset_niveles_mensual.csv')
print('  - dataset_niveles_estandarizado_mensual.csv')
print('  - dataset_transformado_mensual.csv')
print('  - dataset_estandarizado_mensual.csv')
print()
print(f'Comparación de tamaño de muestra:')
print(f'  Trimestral: 72 observaciones x ~18 variables (n/p ≈ 4.5)')
print(f'  Mensual:    {df_estandarizado.shape[0]} observaciones x {df_estandarizado.shape[1]} variables '
      f'(n/p ≈ {df_estandarizado.shape[0]/df_estandarizado.shape[1]:.1f})')

Datasets guardados en data/processed/:
  - dataset_niveles_mensual.csv
  - dataset_niveles_estandarizado_mensual.csv
  - dataset_transformado_mensual.csv
  - dataset_estandarizado_mensual.csv

Comparación de tamaño de muestra:
  Trimestral: 72 observaciones x ~18 variables (n/p ≈ 4.5)
  Mensual:    208 observaciones x 17 variables (n/p ≈ 12.2)
